apapla split karun ghya cells

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# =====================================================
# 1. LOAD DATASET
# =====================================================

file_path = "dataset.csv"   # change dataset name

df = pd.read_csv(file_path)

print("Dataset Loaded Successfully")
print(df.head())

# =====================================================
# 2. HANDLE MISSING VALUES
# =====================================================

# Numerical columns
for col in df.select_dtypes(include=np.number).columns:
    df[col] = df[col].fillna(df[col].mean())

# Categorical columns
for col in df.select_dtypes(exclude=np.number).columns:
    df[col] = df[col].fillna(df[col].mode()[0])

# =====================================================
# 3. ENCODE CATEGORICAL COLUMNS
# =====================================================

encoder = LabelEncoder()

for col in df.select_dtypes(exclude=np.number).columns:
    df[col] = encoder.fit_transform(df[col])

# =====================================================
# 4. SELECT TARGET COLUMN
# =====================================================

target_column = df.columns[-1]   # last column as target

X = df.drop(columns=[target_column])
y = df[target_column]

# =====================================================
# 5. NORMALIZATION
# =====================================================

scaler = StandardScaler()

X = scaler.fit_transform(X)

# =====================================================
# 6. TRAIN TEST SPLIT
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# =====================================================
# 7. CREATE CLIENT DATASETS
# =====================================================

num_clients = 3

client_data = np.array_split(X_train, num_clients)
client_labels = np.array_split(y_train, num_clients)

print(f"\nDataset divided among {num_clients} clients")

# =====================================================
# 8. LOCAL TRAINING
# =====================================================

client_weights = []

for i in range(num_clients):

    print(f"\nTraining Client {i+1}")

    # Local model
    model = LogisticRegression(max_iter=1000)

    # Train locally
    model.fit(client_data[i], client_labels[i])

    # Save weights
    client_weights.append(model.coef_)

    print("Local Training Completed")

# =====================================================
# 9. FEDERATED AVERAGING (FedAvg)
# =====================================================

global_weights = np.mean(client_weights, axis=0)

print("\nFederated Averaging Completed")

# =====================================================
# 10. CREATE GLOBAL MODEL
# =====================================================

global_model = LogisticRegression(max_iter=1000)

# Initialize model
global_model.fit(X_train, y_train)

# Replace local weights with federated weights
global_model.coef_ = global_weights

# =====================================================
# 11. TEST GLOBAL MODEL
# =====================================================

predictions = global_model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("\nGlobal Model Accuracy:", accuracy)